In [37]:
import json

DRAFTS_PATH = "data/drafts/drafts.json"

def load_drafts():
    with open(DRAFTS_PATH) as f:
        return json.load(f)

drafts = load_drafts()
print("Sections:", list(drafts.keys()))


Sections: ['abstracts', 'methods_comparison', 'results_synthesis', 'apa_references']


In [38]:
def evaluate_quality(text, target_min=80, target_max=150):
    wc = len(text.split())
    if wc < target_min:
        return "Too short — expand key points (context, method, findings, significance)."
    if wc > target_max:
        return "Too long — condense, remove repetition, keep core claims and results."
    return "Length is good."

def revise_text(text):
    # Light polish: trim whitespace, normalize spaces, remove obvious filler words
    cleaned = " ".join(text.split())
    for filler in ["very", "actually", "really", "basically", "in order to"]:
        cleaned = cleaned.replace(filler + " ", "")
    return cleaned

def critique_and_revise_drafts(drafts):
    revised = {}
    notes = {}
    # Abstracts
    revised_abs = []
    notes_abs = []
    for abs_text in drafts.get("abstracts", []):
        revised_abs.append(revise_text(abs_text))
        notes_abs.append(evaluate_quality(abs_text))
    revised["abstracts"] = revised_abs
    notes["abstracts_notes"] = notes_abs

    # Methods comparison
    mc = drafts.get("methods_comparison", "")
    revised["methods_comparison"] = revise_text(mc)
    notes["methods_notes"] = evaluate_quality(mc)

    # Results synthesis
    rs = drafts.get("results_synthesis", "")
    revised["results_synthesis"] = revise_text(rs)
    notes["results_notes"] = evaluate_quality(rs)

    # APA references (keep as-is)
    revised["apa_references"] = drafts.get("apa_references", "No references provided.")

    return revised, notes

revised, notes = critique_and_revise_drafts(drafts)
print("Abstract note:", notes["abstracts_notes"][0] if notes["abstracts_notes"] else "No abstracts")


Abstract note: Too short — expand key points (context, method, findings, significance).


In [42]:
!pip -q install gradio

import gradio as gr
import json

DRAFTS_PATH = "data/drafts/drafts.json"

def load_drafts():
    with open(DRAFTS_PATH) as f:
        return json.load(f)

def show_current():
    d = load_drafts()
    return (
        "\n\n---\n".join(d.get("abstracts", [])) or "No abstracts",
        d.get("methods_comparison", "No methods comparison"),
        d.get("results_synthesis", "No results synthesis"),
        d.get("apa_references", "No references")
    )

def critique_and_revise_drafts(drafts):
    def evaluate_quality(text, target_min=80, target_max=150):
        wc = len(text.split())
        if wc < target_min:
            return "🟡 Too short — expand key points (context, method, findings, significance)."
        if wc > target_max:
            return "🔴 Too long — condense, remove repetition, keep core claims and results."
        return "🟢 Length is good."

    def revise_text(text):
        cleaned = " ".join(text.split())
        for filler in ["very", "actually", "really", "basically", "in order to"]:
            cleaned = cleaned.replace(filler + " ", "")
        return cleaned

    revised = {}
    notes = {}

    revised_abs = []
    notes_abs = []
    for abs_text in drafts.get("abstracts", []):
        revised_abs.append(revise_text(abs_text))
        notes_abs.append(evaluate_quality(abs_text))
    revised["abstracts"] = revised_abs
    notes["abstracts_notes"] = notes_abs

    mc = drafts.get("methods_comparison", "")
    revised["methods_comparison"] = revise_text(mc)
    notes["methods_notes"] = evaluate_quality(mc)

    rs = drafts.get("results_synthesis", "")
    revised["results_synthesis"] = revise_text(rs)
    notes["results_notes"] = evaluate_quality(rs)

    revised["apa_references"] = drafts.get("apa_references", "No references provided.")
    return revised, notes

def critique_and_revise():
    d = load_drafts()
    revised, notes = critique_and_revise_drafts(d)
    return (
        "\n\n---\n".join(revised.get("abstracts", [])),
        revised.get("methods_comparison", ""),
        revised.get("results_synthesis", ""),
        revised.get("apa_references", ""),
        "\n".join(notes.get("abstracts_notes", [])),
        notes.get("methods_notes", ""),
        notes.get("results_notes", "")
    )

with gr.Blocks(title="Milestone 4 UI") as demo:
    gr.Markdown("<h1 style='color:#4CAF50;'>🌟 Milestone 4: Review, Refine, and Present Drafts</h1>")
    gr.Markdown("<p style='color:#555;font-size:16px;'>Use the buttons below to load your generated content and apply critique/refinement. Each section is color-coded for clarity.</p>")

    with gr.Row():
        abs_box = gr.Textbox(label="🟦 Abstracts", lines=10, show_copy_button=True)
        mc_box = gr.Textbox(label="🟨 Methods Comparison", lines=10, show_copy_button=True)
    with gr.Row():
        rs_box = gr.Textbox(label="🟩 Results Synthesis", lines=10, show_copy_button=True)
        ref_box = gr.Textbox(label="🟥 APA References", lines=10, show_copy_button=True)

    gr.Markdown("<h3 style='color:#2196F3;'>📝 Critique Notes</h3>")

    with gr.Row():
        abs_notes = gr.Textbox(label="🟦 Abstract Notes", lines=6)
        mc_notes = gr.Textbox(label="🟨 Methods Notes", lines=6)
        rs_notes = gr.Textbox(label="🟩 Results Notes", lines=6)

    with gr.Row():
        load_btn = gr.Button("📂 Load Drafts")
        revise_btn = gr.Button("🔧 Critique & Revise")

    load_btn.click(fn=show_current, inputs=[], outputs=[abs_box, mc_box, rs_box, ref_box])
    revise_btn.click(fn=critique_and_revise, inputs=[], outputs=[abs_box, mc_box, rs_box, ref_box, abs_notes, mc_notes, rs_notes])

demo.launch(share=False)



Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

In [43]:
import os, json

FINAL_DIR = "data/final"
os.makedirs(FINAL_DIR, exist_ok=True)

def build_final_report():
    d = load_drafts()
    revised, notes = critique_and_revise_drafts(d)
    report = []
    report.append("Title: Draft Research Report")
    report.append("\nAbstracts:\n" + "\n\n---\n".join(revised.get("abstracts", [])))
    report.append("\nMethods Comparison:\n" + revised.get("methods_comparison", ""))
    report.append("\nResults Synthesis:\n" + revised.get("results_synthesis", ""))
    report.append("\nAPA References:\n" + revised.get("apa_references", ""))
    return "\n".join(report)

final_text = build_final_report()
with open(os.path.join(FINAL_DIR, "final_report.txt"), "w") as f:
    f.write(final_text)

print("Saved:", os.path.join(FINAL_DIR, "final_report.txt"))


Saved: data/final/final_report.txt
